# libm_diff — which maths functions agree with C?

Samples 200,000 inputs per maths function and dumps raw bits, so the C and Rust results can be compared exactly. This is the measurement that underpins the whole project: sin, cos, tan, atan2, sqrt, fmod, exp, log and cbrt are bit-identical to Microsoft's C library, and pow is the one exception.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example libm_diff
cd porttest
../target/release/examples/libm_diff
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "libm_diff"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
import struct

def unbits(h):
    """Turn a 16-hex-digit IEEE-754 bit pattern back into a float."""
    return struct.unpack("<d", int(h, 16).to_bytes(8, "little"))[0]

def read_state(path):
    """Read one of the raw-bit state dumps into {label: [floats]}."""
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if not parts:
                continue
            key, rest = parts[0], parts[1:]
            vals = []
            for tok in rest:
                if len(tok) == 16:
                    try:
                        vals.append(unbits(tok))
                        continue
                    except ValueError:
                        pass
                vals.append(tok)
            out.setdefault(key, []).append(vals)
    return out

def compare(a, b, label_a="C", label_b="Rust"):
    """Byte-compare two dump files and report."""
    ta = open(a, "rb").read().replace(b"\r\n", b"\n")
    tb = open(b, "rb").read().replace(b"\r\n", b"\n")
    if ta == tb:
        print(f"BIT-IDENTICAL: {label_a} and {label_b} agree on every bit")
        return True
    print(f"MISMATCH between {label_a} and {label_b}")
    la, lb = ta.decode().splitlines(), tb.decode().splitlines()
    for i, (x, y) in enumerate(zip(la, lb)):
        if x != y:
            print(f"  line {i}:\n    {label_a}: {x}\n    {label_b}: {y}")
    return False


In [3]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

done



In [4]:
c  = os.path.join(WORK, "libm_c.txt")
rs = os.path.join(WORK, "libm_rust.txt")
if os.path.exists(c) and os.path.exists(rs):
    lc, lr = open(c).read().splitlines(), open(rs).read().splitlines()
    from collections import Counter
    bad = Counter()
    for a, b in zip(lc, lr):
        if a != b:
            bad[a.split()[0]] += 1
    print(f"compared {len(lc)} samples")
    print("functions that differ:", dict(bad) if bad else "NONE - all bit-identical")
else:
    print("(run both libm_diff programs in porttest/ to compare)")


compared 200000 samples
functions that differ: NONE - all bit-identical
